# 01 CTU-13  Dataset Conversion (Python)

* Extract columns.
* Write two files:
* ctu_edges.csv: for GraphSketch
* ctu_labels.csv: for evaluation

In [ ]:
import pandas as pd

# Load raw CTU data (assuming whitespace-separated and no header)
df = pd.read_csv("data/ctu_dataset/ctu_modified.txt",
                 delim_whitespace=True, header=None)
df.columns = ["timestamp", "src", "dst", "label"]
print(df.head())

# Save edge list as expected by GraphSketch (src,dst,timestamp)
df[["src", "dst", "timestamp"]].to_csv(
    "data/ctu_dataset/ctu_edges.csv", index=False, header=False)


# Save labels separately (1 for botnet, 0 for normal)
df["label"].to_csv("data/ctu_dataset/ctu_labels.csv", index=False, header=False)

print("✅ CTU-13 conversion complete.")
print("- Edges saved to: data/ctu_dataset/ctu_edges.csv")
print("- Labels saved to: data/ctu_dataset/ctu_labels.csv")

df_new = pd.read_csv("data/ctu_dataset/ctu_edges.csv", delim_whitespace=True, header =None)
print(df_new.head())

df_label = pd.read_csv("data/ctu_dataset/ctu_labels.csv",
                     delim_whitespace=True, header=None)
print(df_label.head())

/tmp/ipykernel_366984/3593013624.py:4: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv("data/ctu_dataset/ctu_modified.txt",


   timestamp  src  dst  label
0          1   71   59      0
1          2  245   59      1
2          3  139   59      0
3          4   64   59      0
4          5  166  110      1
✅ CTU-13 conversion complete.
- Edges saved to: data/ctu_dataset/ctu_edges.csv
- Labels saved to: data/ctu_dataset/ctu_labels.csv
           0
0    71,59,1
1   245,59,2
2   139,59,3
3    64,59,4
4  166,110,5


/tmp/ipykernel_366984/3593013624.py:21: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df_new = pd.read_csv("data/ctu_dataset/ctu_edges.csv", delim_whitespace=True, header =None)
/tmp/ipykernel_366984/3593013624.py:24: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df_label = pd.read_csv("data/ctu_dataset/ctu_labels.csv",


   0
0  0
1  1
2  0
3  0
4  1


In [4]:
print("Raw ctu-modified:", len(df))
print("new edges ctu: ", len(df_new))
print("ctu label", len(df_label))

Raw ctu-modified: 1416971
new edges ctu:  1416971
ctu label 1416971


# 02 CIC-IDS2017 

* Convert the CIC-IDS2017 flows into a dynamic graph edge stream that GraphSketch can process:

* `timestamp, src_id, dst_id, label`
   - timestamp = flow start time (or rounded time window)

    - src_id / dst_id = IPs or port-based identifiers (mapped to integers)

    - label = 1 for anomaly, 0 for normal

In [11]:
import glob
import pandas as pd
import os

def convert_cic_ids_to_edges(input_folder, output_edges_path, output_labels_path):
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
    print(f"Processing {len(csv_files)} CSV files from {input_folder}...")

    edges = []
    labels = []

    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file, low_memory=False)
            df.columns = df.columns.str.strip()  # Fix whitespace

            for idx, row in df.iterrows():
                src = hash(row["Source IP"]) % 500
                dst = hash(row["Destination IP"]) % 500
                timestamp = idx // 1000  # adjust granularity if needed
                label = int(row["Label"])

                edges.append(f"{timestamp},{src},{dst}")
                labels.append(label)
        except Exception as e:
            print(f"⚠️ Failed to process {csv_file}: {e}")

    with open(output_edges_path, "w") as f:
        for edge in edges:
            f.write(edge + "\n")

    with open(output_labels_path, "w") as f:
        for label in labels:
            f.write(str(label) + "\n")

    print(f"✅ Saved {len(edges)} edges to {output_edges_path}")
    print(f"✅ Saved {len(labels)} labels to {output_labels_path}")


In [12]:
convert_cic_ids_to_edges(
    "data/CIC_IDS2017/MachineLearningCSV/MachineLearningCVE",
    "data/CIC_IDS2017/cic_edges.txt",
    "data/CIC_IDS2017/cic_ground_truth.csv"
)


Processing 0 CSV files from data/CIC_IDS2017/MachineLearningCSV/MachineLearningCVE...
✅ Saved 0 edges to data/CIC_IDS2017/cic_edges.txt
✅ Saved 0 labels to data/CIC_IDS2017/cic_ground_truth.csv


In [5]:
!pwd

/home/oaekle42@tntech.edu/Desktop/GraphSketch/Graph-Sketch


In [29]:
import pandas as pd

df = pd.read_csv("data/CIC_IDS2017/MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
print(df.columns.tolist())


[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Count', ' URG Flag 

In [30]:
import pandas as pd

def convert_tuesday_flow_to_graph():
    path = "data/CIC_IDS2017/MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
    edge_out = "data/CIC_IDS2017/friday_edges.txt"
    label_out = "data/CIC_IDS2017/friday_labels.csv"

    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()

    if "Destination Port" not in df.columns or "Flow Duration" not in df.columns or "Label" not in df.columns:
        print("❌ Required columns not found.")
        print(df.columns.tolist())
        return

    edges, labels = [], []

    for idx, row in df.iterrows():
        try:
            src = hash(row["Destination Port"]) % 500
            dst = hash(row["Flow Duration"]) % 500
            timestamp = idx // 1000

            label_str = str(row["Label"]).strip().lower()
            label = 0 if label_str == "benign" else 1

            edges.append(f"{timestamp},{src},{dst}")
            labels.append(label)
        except:
            continue

    with open(edge_out, "w") as ef:
        ef.write("\n".join(edges))
    with open(label_out, "w") as lf:
        lf.write("\n".join(map(str, labels)))

    print(f"✅ Converted: {len(edges)} edges written to {edge_out}")
    print(f"✅ Labels: {len(labels)} saved to {label_out}")

# Run it
convert_tuesday_flow_to_graph()


✅ Converted: 225745 edges written to data/CIC_IDS2017/friday_edges.txt
✅ Labels: 225745 saved to data/CIC_IDS2017/friday_labels.csv


In [16]:
df = pd.read_csv("data/CIC_IDS2017/MachineLearningCVE/Tuesday-WorkingHours.pcap_ISCX.csv", low_memory=False)
df.columns = df.columns.str.strip()
print(df["Label"].unique())


['BENIGN' 'FTP-Patator' 'SSH-Patator']


In [25]:
import pandas as pd

# Load files
scores = pd.read_csv("score.txt", header=None)
edges = pd.read_csv("data/CIC_IDS2017/tuesday_edges.txt", header=None)

# Check for NaNs
print("NaNs in scores:", scores.isnull().values.any())
print("NaNs in edges.txt:", edges.isnull().values.any())

print("Scores shape:", scores.shape)
print("Tuesda_edge shape:", edges.shape)


NaNs in scores: True
NaNs in edges.txt: False
Scores shape: (445909, 4)
Tuesda_edge shape: (445909, 3)


In [31]:
import pandas as pd
labels = pd.read_csv("data/CIC_IDS2017/friday_labels.csv", header=None)
print(labels.value_counts())


0
1    128027
0     97718
Name: count, dtype: int64


## Combining all CSVs

In [32]:
import pandas as pd
import os

# Path to your CSV files
folder = "data/CIC_IDS2017/MachineLearningCVE"
output_edge_file = os.path.join(folder, "cic_edges.txt")
output_label_file = os.path.join(folder, "cic_ground_truth.csv")

# Check all files in the directory
files = [f for f in os.listdir(folder) if f.endswith(".csv") and "pcap_ISCX" in f]
print(f"Found {len(files)} CSV files.")

edges_all = []
labels_all = []

for file in files:
    print(f"Processing {file}...")
    try:
        df = pd.read_csv(os.path.join(folder, file), low_memory=False)
        df.columns = df.columns.str.strip()  # Clean column names

        if not {"Destination Port", "Flow Duration", "Label"}.issubset(df.columns):
            print(f"⚠️ Skipping {file} due to missing columns.")
            continue

        for idx, row in df.iterrows():
            try:
                src = hash(row["Destination Port"]) % 500
                dst = hash(row["Flow Duration"]) % 500
                timestamp = idx // 1000
                label = int(row["Label"] != "BENIGN")  # 1 if not benign, else 0

                edges_all.append(f"{timestamp},{src},{dst}")
                labels_all.append(str(label))
            except:
                continue
    except Exception as e:
        print(f"❌ Error reading {file}: {e}")
        continue

# Write combined edges and labels
with open(output_edge_file, "w") as f:
    f.write("\n".join(edges_all))
with open(output_label_file, "w") as f:
    f.write("\n".join(labels_all))

print(f"\n✅ Combined {len(edges_all)} edges saved to: {output_edge_file}")
print(f"✅ Combined {len(labels_all)} labels saved to: {output_label_file}")


Found 8 CSV files.
Processing Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv...
Processing Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv...
Processing Wednesday-workingHours.pcap_ISCX.csv...
Processing Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv...
Processing Monday-WorkingHours.pcap_ISCX.csv...
Processing Friday-WorkingHours-Morning.pcap_ISCX.csv...
Processing Tuesday-WorkingHours.pcap_ISCX.csv...
Processing Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...

✅ Combined 2830743 edges saved to: data/CIC_IDS2017/MachineLearningCVE/cic_edges.txt
✅ Combined 2830743 labels saved to: data/CIC_IDS2017/MachineLearningCVE/cic_ground_truth.csv
